# Line ratios 

In [1]:
import numpy as np
import pandas as pd

from astroExplain.spectra import astronomy
from sdss.metadata import MetaData

meta = MetaData()
pd.set_option("display.max_columns", None)

# Custom functions

# Config

## Constants

In [2]:
se_cols = ["mse", "mse_filter_250", "mse_97", "mse_filter_250_97"]
se_rank_cols = [f"rank_{col}" for col in se_cols]
# ----------------------------------------------
rse_cols = [f"{col}_rel" for col in se_cols]
rse_rank_cols = [f"rank_{col}_rel" for col in se_cols]
# ----------------------------------------------
se_family = ["mse", "mse_97", "mse_filter_250", "mse_filter_250_97"]
rse_family = [f"{col}_rel" for col in se_family]

## Directories

In [3]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

# Data

In [4]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave * 0.1

spectra = np.load(f"{spectra_dir}/spectra_imputed.npy", mmap_mode="r")

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

meta_with_lines_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_lines.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(f"{spectra_dir}/ids_imputing.npy", mmap_mode="r")

# lines_df = pd.read_csv(
#     f"{spectra_dir}/meta_data/lines_EdgarOrtiz.csv",
#     index_col="specObjID",
# )

In [5]:
meta_with_lines_df.head(3)
# final_meta_df.head(3)

,mjd,plate,fiberid,run2d,ra,dec,z,zErr,zWarning,class,subClass,z_noqso,zErr_noqso,zWarning_noqso,targetType,programname,instrument,snMedian,ABSSB,BROAD,ebv,indexArray,oii_3726_flux,oii_3726_flux_err,oii_3729_flux,oii_3729_flux_err,neiii_3869_flux,neiii_3869_flux_err,h_delta_flux,h_delta_flux_err,h_gamma_flux,h_gamma_flux_err,oiii_4363_flux,oiii_4363_flux_err,h_beta_flux,h_beta_flux_err,oiii_4959_flux,oiii_4959_flux_err,oiii_5007_flux,oiii_5007_flux_err,hei_5876_flux,hei_5876_flux_err,oi_6300_flux,oi_6300_flux_err,nii_6548_flux,nii_6548_flux_err,h_alpha_flux,h_alpha_flux_err,nii_6584_flux,nii_6584_flux_err,sii_6717_flux,sii_6717_flux_err,sii_6731_flux,sii_6731_flux_err,ariii7135_flux,ariii7135_flux_err,oii_flux,oii_flux_err,oiii_flux,oiii_flux_err
specobjid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1874756710606858240,52976,1665,485,26,49.861444,41.540485,0.017909,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,94.13022,BROADLINE,BROADLINE,0.138756,2.0,0.0,-7.832294e-01,30418890.00,1.842604e+06,-40.570890,10.948140,21.51809,13.21366,49.39598,15.19424,-77.73644,16.07351,109.63340,17.43791,-49.362380,19.09110,176.81650,17.88523,-201.0114,16.10323,34.87749,22.67058,49.30014,11.300770,109.7575,35.74986,148.70060,34.08569,3.150983,31.61602,-2.93755,31.82735,-103.19390,23.14498,0.0000,1.107654,174.70850,21.45677
1874765781577787392,52976,1665,518,26,49.918970,41.548643,0.021434,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,92.63803,BROADLINE,BROADLINE,0.141073,3.0,-603451.0,1.139662e+06,30924.64,3.962919e+04,-1.672035,7.612301,30.98863,11.73072,26.67140,12.99125,-20.94539,10.47208,41.71867,14.98429,2.058654,12.20163,72.43165,12.41198,-289.6941,13.68605,-13.75883,14.15635,29.28021,6.114465,132.8577,23.17760,88.31586,18.44262,-44.478840,16.03369,18.97761,16.29948,-36.61877,14.79838,722.0763,1365.593000,67.69716,12.80094
1874776226938251264,52976,1665,556,26,50.251707,41.562451,0.015699,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.91905,BROADLINE,BROADLINE,0.142785,5.0,0.0,-7.819291e-01,0.00,-7.819291e-01,-42.426140,9.273634,16.98602,11.56076,51.72350,12.93845,-121.21380,13.62333,169.26810,14.49883,-80.583990,15.25546,147.23940,15.11998,-346.5715,12.84481,21.52090,17.46943,45.43762,7.907939,261.2984,25.16728,137.05040,23.85214,27.470890,23.79437,-16.98146,23.95586,-95.06487,18.50055,0.0000,1.105815,150.70450,14.26691


## Line to metadata

```python
meta_with_lines_df = final_meta_df.join(lines_df, how="left")
meta_with_lines_df.to_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_lines.csv.gz",
    index=True,
    compression="gzip"
)
```

## SE + RSE scores

In [6]:
bin_id = "bin_03"
res_scores_03_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    usecols=["specobjid"] + se_cols + rse_cols,
    index_col="specobjid",
)

rank = np.arange(res_scores_03_df.shape[0]) + 1
res_03_scores_rank_df = res_scores_03_df.copy()
# score_rank_df
for col in res_scores_03_df.columns:

    index_sorted = res_scores_03_df.sort_values(by=col, ascending=False).index

    res_03_scores_rank_df.loc[index_sorted, f"rank_{col}"] = rank
    res_03_scores_rank_df[f"rank_{col}"].astype(int)

n_spec = res_scores_03_df.shape[0]
n_top_1_pct = int(n_spec * 0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [7]:
res_03_scores_rank_df.head()

,mse_rel,mse_filter_250_97_rel,mse_97_rel,mse,mse_97,mse_filter_250_97,mse_filter_250,mse_filter_250_rel,rank_mse_rel,rank_mse_filter_250_97_rel,rank_mse_97_rel,rank_mse,rank_mse_97,rank_mse_filter_250_97,rank_mse_filter_250,rank_mse_filter_250_rel
specobjid,,,,,,,,,,,,,,,,
1437819582281705472,3.549991,2.582845,2.660625,3.570211,2.631072,2.548220,3.305904,3.388511,13249.0,21423.0,20366.0,8755.0,10313.0,10084.0,7623.0,15236.0
1792473383787587584,3.592197,2.827291,2.874075,3.230481,2.715585,2.662638,3.150359,3.490994,11806.0,6400.0,7507.0,16243.0,7376.0,6116.0,11194.0,11277.0
1566270578824865792,3.799695,2.967733,3.016445,3.215022,2.564632,2.517923,3.124369,3.686568,6946.0,3265.0,3952.0,16789.0,13498.0,11560.0,11972.0,6498.0
1833056640966879232,3.418285,2.751345,2.802071,2.965650,2.507619,2.460364,2.933262,3.354621,18889.0,9324.0,10444.0,28087.0,16967.0,14928.0,20074.0,16849.0
2824940385879484416,3.393606,2.592578,2.649388,2.839407,2.384083,2.344867,2.787019,3.352981,20228.0,20498.0,21431.0,36848.0,27263.0,24506.0,29881.0,16922.0


## Lines to Bin sample

In [8]:
res_03_scores_lines_df = res_03_scores_rank_df.join(meta_with_lines_df, how="inner")
print(res_03_scores_lines_df.shape, res_03_scores_rank_df.shape)
res_03_scores_lines_df.columns

(181850, 76) (181850, 16)


Index(['mse_rel', 'mse_filter_250_97_rel', 'mse_97_rel', 'mse', 'mse_97',
       'mse_filter_250_97', 'mse_filter_250', 'mse_filter_250_rel',
       'rank_mse_rel', 'rank_mse_filter_250_97_rel', 'rank_mse_97_rel',
       'rank_mse', 'rank_mse_97', 'rank_mse_filter_250_97',
       'rank_mse_filter_250', 'rank_mse_filter_250_rel', 'mjd', 'plate',
       'fiberid', 'run2d', 'ra', 'dec', 'z', 'zErr', 'zWarning', 'class',
       'subClass', 'z_noqso', 'zErr_noqso', 'zWarning_noqso', 'targetType',
       'programname', 'instrument', 'snMedian', 'ABSSB', 'BROAD', 'ebv',
       'indexArray', 'oii_3726_flux', 'oii_3726_flux_err', 'oii_3729_flux',
       'oii_3729_flux_err', 'neiii_3869_flux', 'neiii_3869_flux_err',
       'h_delta_flux', 'h_delta_flux_err', 'h_gamma_flux', 'h_gamma_flux_err',
       'oiii_4363_flux', 'oiii_4363_flux_err', 'h_beta_flux',
       'h_beta_flux_err', 'oiii_4959_flux', 'oiii_4959_flux_err',
       'oiii_5007_flux', 'oiii_5007_flux_err', 'hei_5876_flux',
       'hei_5

# EDA fluxes

In [9]:
lines_cols = [
    "oii_3726_flux",
    "oii_3729_flux",
    "h_beta_flux",
    "oiii_4959_flux",
    "oiii_5007_flux",
    "oiii_flux",
    "h_alpha_flux",
    "nii_6548_flux",
    "nii_6584_flux",
    "sii_6717_flux",
    "sii_6731_flux",
]
res_03_scores_lines_df[lines_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
oii_3726_flux,181457.0,7.000327e+06,1.463272e+09,-1.796925e+11,2.301679,14.878620,39.01466,3.091597e+11
oii_3729_flux,181457.0,2.079458e+07,3.333411e+09,-2.694684e+11,1.622084,14.811510,38.22337,1.087276e+12
h_beta_flux,181457.0,2.319494e+05,2.735681e+07,-5.777803e+06,4.933405,12.890140,43.42221,6.261624e+09
oiii_4959_flux,181457.0,6.330987e+04,9.658346e+06,-1.476749e+08,0.571599,4.736154,10.96004,2.556557e+09
oiii_5007_flux,181457.0,1.538721e+05,2.334485e+07,-6.166620e+06,7.543362,15.627500,31.71705,7.612909e+09
oiii_flux,181457.0,1.567924e+05,2.345455e+07,-6.328831e+06,7.938534,16.213250,32.66922,7.590690e+09
h_alpha_flux,181457.0,8.975622e+05,1.051919e+08,-9.832831e+08,13.856210,37.800220,179.15010,2.292654e+10
nii_6548_flux,181457.0,4.411457e+05,1.213879e+08,-2.264222e+07,3.133515,10.877480,34.11325,5.080869e+10
nii_6584_flux,181457.0,1.330598e+06,3.661340e+08,-6.829417e+07,9.451404,32.809000,102.89340,1.532508e+11
sii_6717_flux,181457.0,-1.246196e+06,6.185059e+08,-2.631785e+11,2.217676,12.849050,42.71759,9.422665e+09


In [10]:
# Compute deciles (10th to 90th percentiles) for each column
deciles = np.arange(0, 1.01, 0.1)
decile_df = res_03_scores_lines_df[lines_cols].quantile(q=deciles).T
decile_df.columns = [f"{int(q * 100)}th" for q in deciles]
decile_df.T

,oii_3726_flux,oii_3729_flux,h_beta_flux,oiii_4959_flux,oiii_5007_flux,oiii_flux,h_alpha_flux,nii_6548_flux,nii_6584_flux,sii_6717_flux,sii_6731_flux
0th,-1.796925e+11,-2.694684e+11,-5.777803e+06,-1.476749e+08,-6.166620e+06,-6.328831e+06,-9.832831e+08,-2.264222e+07,-6.829417e+07,-2.631785e+11,-7.180114e+10
10th,-3.117972e+00,-5.607386e+00,1.076345e+00,-3.381962e+00,2.381581e+00,2.896852e+00,5.333937e+00,5.047289e-01,1.522379e+00,-2.222454e+00,-3.537128e+00
20th,2.400316e-01,0.000000e+00,3.731005e+00,-3.495984e-01,5.882900e+00,6.321469e+00,1.063953e+01,2.036395e+00,6.142238e+00,8.097096e-01,-3.707439e-01
30th,4.466964e+00,3.866060e+00,6.216536e+00,1.438902e+00,9.135857e+00,9.518421e+00,1.735361e+01,4.378261e+00,1.320585e+01,3.853062e+00,1.924226e+00
40th,9.320536e+00,8.996651e+00,9.095771e+00,3.072225e+00,1.223833e+01,1.267844e+01,2.555636e+01,7.218415e+00,2.177240e+01,7.808183e+00,4.786159e+00
50th,1.487862e+01,1.481151e+01,1.289014e+01,4.736154e+00,1.562750e+01,1.621325e+01,3.780022e+01,1.087748e+01,3.280900e+01,1.284905e+01,8.500538e+00
60th,2.185441e+01,2.170752e+01,1.889692e+01,6.649740e+00,1.991223e+01,2.065898e+01,6.121434e+01,1.670178e+01,5.037641e+01,2.012888e+01,1.388159e+01
70th,3.179890e+01,3.132968e+01,3.169217e+01,9.189840e+00,2.648865e+01,2.737358e+01,1.192963e+02,2.654161e+01,8.005562e+01,3.280434e+01,2.331403e+01
80th,4.926152e+01,4.784514e+01,6.196866e+01,1.359348e+01,3.965854e+01,4.073356e+01,2.667649e+02,4.425043e+01,1.334695e+02,5.668757e+01,4.162605e+01
90th,9.473451e+01,9.172521e+01,1.330529e+02,2.836828e+01,8.511094e+01,8.720853e+01,5.823152e+02,8.112327e+01,2.446865e+02,1.080476e+02,8.074018e+01


In [11]:
n_nans = res_03_scores_lines_df[lines_cols].isna().sum()
n_negatives = (res_03_scores_lines_df[lines_cols] < 0).sum()
n_zeros = (res_03_scores_lines_df[lines_cols] == 0).sum()
temp_df = pd.DataFrame(
    {"n_nans": n_nans, "n_negatives": n_negatives, "n_zeros": n_zeros}
)
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_nans,n_negatives,n_zeros
0,oii_3726_flux,393,29249,5864
1,oii_3729_flux,393,33043,5332
2,h_beta_flux,393,12432,280
3,oiii_4959_flux,393,39494,278
4,oiii_5007_flux,393,7368,280
5,oiii_flux,393,7688,273
6,h_alpha_flux,393,3293,502
7,nii_6548_flux,393,10664,420
8,nii_6584_flux,393,10664,420
9,sii_6717_flux,393,30392,535


# Data prep to compute ratios

In [12]:
# replace 0 with NaN
res_03_scores_lines_df[lines_cols] = res_03_scores_lines_df[lines_cols].replace(
    0, np.nan
)
# replace negative fluxes numeric values with NaN
res_03_scores_lines_df[lines_cols] = res_03_scores_lines_df[lines_cols].where(
    res_03_scores_lines_df[lines_cols] >= 0, np.nan
)
n_zeros = (res_03_scores_lines_df[lines_cols] == 0).sum()
n_negatives = (res_03_scores_lines_df[lines_cols] < 0).sum()
temp_df = pd.DataFrame({"n_negatives": n_negatives, "n_zeros": n_zeros})
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_negatives,n_zeros
0,oii_3726_flux,0,0
1,oii_3729_flux,0,0
2,h_beta_flux,0,0
3,oiii_4959_flux,0,0
4,oiii_5007_flux,0,0
5,oiii_flux,0,0
6,h_alpha_flux,0,0
7,nii_6548_flux,0,0
8,nii_6584_flux,0,0
9,sii_6717_flux,0,0


# Line ratios

In [13]:
ratios_df, cols_ratios = astronomy.compute_emission_line_ratios(res_03_scores_lines_df)
ratios_df.head()

,mse_rel,mse_filter_250_97_rel,mse_97_rel,mse,mse_97,mse_filter_250_97,mse_filter_250,mse_filter_250_rel,rank_mse_rel,rank_mse_filter_250_97_rel,rank_mse_97_rel,rank_mse,rank_mse_97,rank_mse_filter_250_97,rank_mse_filter_250,rank_mse_filter_250_rel,mjd,plate,fiberid,run2d,ra,dec,z,zErr,zWarning,class,subClass,z_noqso,zErr_noqso,zWarning_noqso,targetType,programname,instrument,snMedian,ABSSB,BROAD,ebv,indexArray,oii_3726_flux,oii_3726_flux_err,oii_3729_flux,oii_3729_flux_err,neiii_3869_flux,neiii_3869_flux_err,h_delta_flux,h_delta_flux_err,h_gamma_flux,h_gamma_flux_err,oiii_4363_flux,oiii_4363_flux_err,h_beta_flux,h_beta_flux_err,oiii_4959_flux,oiii_4959_flux_err,oiii_5007_flux,oiii_5007_flux_err,hei_5876_flux,hei_5876_flux_err,oi_6300_flux,oi_6300_flux_err,nii_6548_flux,nii_6548_flux_err,h_alpha_flux,h_alpha_flux_err,nii_6584_flux,nii_6584_flux_err,sii_6717_flux,sii_6717_flux_err,sii_6731_flux,sii_6731_flux_err,ariii7135_flux,ariii7135_flux_err,oii_flux,oii_flux_err,oiii_flux,oiii_flux_err,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
specobjid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1437819582281705472,3.549991,2.582845,2.660625,3.570211,2.631072,2.548220,3.305904,3.388511,13249.0,21423.0,20366.0,8755.0,10313.0,10084.0,7623.0,15236.0,52765,1277,165,26,148.46827,38.172606,0.031511,0.000007,0,GALAXY,STARFORMING,0,0,0,SCIENCE,legacy,SDSS,17.54371,STARFORMING,STARFORMING,0.014841,116506.0,131.223200,6.858774,128.431600,7.103514,7.755666,3.699379,40.485380,3.263404,73.616520,3.243565,0.606630,2.677816,170.202900,3.723063,28.970400,2.676860,74.711250,3.121359,19.887950,2.286599,17.802030,2.272578,67.421100,1.092263,669.678500,7.192680,203.357600,3.294514,121.926300,2.996880,90.05820,2.797693,7.585026,2.344169,256.730200,11.777070,76.773120,3.482531,3.934589,0.303665,0.438954,0.569345,0.160025,0.316547,1.353861
1792473383787587584,3.592197,2.827291,2.874075,3.230481,2.715585,2.662638,3.150359,3.490994,11806.0,6400.0,7507.0,16243.0,7376.0,6116.0,11194.0,11277.0,52990,1592,148,26,140.00282,32.969776,0.050193,0.000020,0,GALAXY,NaN,0,0,0,SCIENCE,legacy,SDSS,17.54372,undefined,undefined,0.015621,116464.0,4.099574,3.181240,9.271741,3.251889,6.308383,2.406566,2.380959,1.574557,8.188803,1.552840,-1.816201,1.590314,9.438538,1.345372,0.255657,1.300283,3.868944,1.389464,5.424542,1.521104,0.742763,1.133368,4.318114,0.489524,35.856190,1.600355,13.024430,1.476515,4.732165,1.222621,4.87282,1.298028,0.907843,1.253644,13.600400,4.315449,9.062179,3.244147,3.798914,0.363241,0.409909,0.943743,0.052493,0.267875,0.971135
1566270578824865792,3.799695,2.967733,3.016445,3.215022,2.564632,2.517923,3.124369,3.686568,6946.0,3265.0,3952.0,16789.0,13498.0,11560.0,11972.0,6498.0,52817,1391,523,26,239.37038,29.083480,0.077171,0.000021,0,GALAXY,NaN,0,0,0,SCIENCE,legacy,SDSS,17.54373,undefined,undefined,0.031838,116418.0,NaN,9.113636,12.939280,9.132084,-6.764836,4.841925,-2.237487,3.003779,5.751787,2.911517,2.215967,3.743118,3.820069,2.522194,NaN,3.673122,2.117211,3.504975,-0.202543,2.568233,-3.942734,3.066033,NaN,1.035421,5.409146,2.507887,NaN,3.123065,12.443120,4.653118,NaN,4.730724,-4.969798,3.635871,-4.060645,5.168249,6.602443,3.662050,1.415981,NaN,0.554234,NaN,NaN,NaN,NaN
1833056640966879232,3.418285,2.751345,2.802071,2.965650,2.507619,2.460364,2.933262,3.354621,18889.0,9324.0,10444.0,28087.0,16967.0,14928.0,20074.0,16849.0,53474,1628,333,26,188.28053,8.365470,0.101359,0.000025,0,GALAXY,NaN,0,0,0,SCIENCE,legacy,SDSS,17.54374,undefined,undefined,0.016053,116376.0,NaN,2.805713,2.097260,2.948378,2.943654,2.305244,-1.690971,1.447742,1.087122,1.514700,3.560174,2.743205,4.580916,1.703570,NaN,2.535907,0.482871,2.156454,0.891663,1.563283,-4.304588,2.103192,0.477401,0.721048,2.375893,1.375433,1.439953,2.174847,NaN,3.255430,NaN,2.008540,2.144301,3.464672,-4.017598,8.294394,NaN,1.528086,0.518650,0.606068,0.105409,NaN,-0.759642,NaN,NaN
2824940385879484416,3.393606,2.592578,2.64

In [14]:
# Compute percentiles (45th to 55th percentiles) for each column
percentiles = np.arange(0.45, 0.55, 0.05)
# Create a DataFrame of percentiles for each column in fluxes_df[lines_cols]
ratios_percentiles_df = ratios_df[cols_ratios].quantile(q=percentiles).T
# Optionally rename the rows as "10th", "20th", etc.
ratios_percentiles_df.columns = [f"{int(q * 100)}th" for q in percentiles]
# Display the result
ratios_percentiles_df

,45th,50th,55th
balmer_decrement,3.548007,3.744923,3.932153
nii_to_halpha,0.583414,0.645449,0.715934
oiii_to_hbeta,0.962152,1.100840,1.249103
oiii_to_oii,0.766708,0.838865,0.919984
o3n2_index,0.117505,0.162499,0.210003
sii_to_halpha,0.380453,0.416128,0.459978
sii_density_ratio,1.354095,1.383720,1.413056


# Common anomalies

In [15]:
overview_common_dict = {
    # row 1
    "narrow_line": 734111514142730240,
    "broad_line_large_OIII": 1633733043925575680,
    # row 2
    "broad_emission_dips_OIII_half": 1192355533059811328,
    "star_forming_step_blue_slope": 1959124163192973312,
    # row 3
    "blue_bump_emission": 531492683672217600,
    "passive_star": 1780176998165932032,
    # row 4
    "spike": 1413149843194406912,
    "noise_forest": 637325355518027776,
}

In [16]:
cols_ratios = [
    "balmer_decrement",
    "nii_to_halpha",
    "oiii_to_hbeta",
    "oiii_to_oii",
    "o3n2_index",
    "sii_to_halpha",
    "sii_density_ratio",
]
ratios_df.loc[734111514142730240, cols_ratios]

balmer_decrement      3.29336
nii_to_halpha         0.05016
oiii_to_hbeta        3.990424
oiii_to_oii               NaN
o3n2_index           1.900664
sii_to_halpha        0.163502
sii_density_ratio    1.362553
Name: 734111514142730240, dtype: object

In [17]:
ratios_dict = {
    "specobjid": [],
    "name": [],
    "balmer_decrement": [],
    "nii_to_halpha": [],
    "oiii_to_hbeta": [],
    "oiii_to_oii": [],
    "o3n2_index": [],
    "sii_to_halpha": [],
    "sii_density_ratio": [],
}


for title, specid in overview_common_dict.items():
    ratios_dict["name"].append(title)
    ratios_dict["specobjid"].append(specid)
    for col in cols_ratios:
        try:
            ratios_dict[col].append(ratios_df.loc[specid, col])
        except KeyError:
            ratios_dict[col].append(np.nan)

In [18]:
common_anomalies_ratios_df = pd.DataFrame(ratios_dict)
common_anomalies_ratios_df.to_clipboard()
common_anomalies_ratios_df

,specobjid,name,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
0,734111514142730240,narrow_line,3.293360,0.050160,3.990424,NaN,1.900664,0.163502,1.362553
1,1633733043925575680,broad_line_large_OIII,3.891876,0.204359,10.550430,11.752099,1.712876,0.217171,1.109206
2,1192355533059811328,broad_emission_dips_OIII_half,5.013954,0.599002,8.754367,7.460670,1.164797,0.344767,1.184883
3,1959124163192973312,star_forming_step_blue_slope,3.437888,0.182478,1.409797,0.595381,0.887945,0.278217,1.376201
4,531492683672217600,blue_bump_emission,3.584317,0.371369,0.517071,0.429446,0.143744,0.333807,1.389607
5,1780176998165932032,passive_star,NaN,NaN,NaN,0.123804,NaN,NaN,NaN
6,1413149843194406912,spike,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,637325355518027776,noise_forest,5.763929,0.055846,0.861052,0.411711,1.188040,NaN,NaN


In [19]:
class_common_anomalies_dict = {
    "specobjid": [],
    "name": [],
    "class": [],
}

for title, specid in overview_common_dict.items():

    class_common_anomalies_dict["name"].append(title)
    class_common_anomalies_dict["specobjid"].append(specid)
    try:
        class_ = final_meta_df.loc[specid, "subClass"]
        class_common_anomalies_dict["class"].append(class_)
    except KeyError:
        class_common_anomalies_dict["class"].append(np.nan)

class_common_anomalies_df = pd.DataFrame(class_common_anomalies_dict)
class_common_anomalies_df.to_clipboard()
class_common_anomalies_df

,specobjid,name,class
0,734111514142730240,narrow_line,STARBURST
1,1633733043925575680,broad_line_large_OIII,STARBURST
2,1192355533059811328,broad_emission_dips_OIII_half,AGN BROADLINE
3,1959124163192973312,star_forming_step_blue_slope,STARBURST
4,531492683672217600,blue_bump_emission,STARFORMING
5,1780176998165932032,passive_star,NaN
6,1413149843194406912,spike,NaN
7,637325355518027776,noise_forest,NaN
